# 12_master_pipeline_runner.ipynb

This notebook orchestrates the entire **BTCUSDT Quantitative Trading Research Framework** end-to-end.

### Objectives:
1. Optionally pull the latest code from GitHub before execution.
2. Automatically verify and install missing dependencies from `requirements.txt`.
3. Define execution flags for each phase of the pipeline.
4. Execute the data downloader, cleaning, feature engineering, labeling, model training, validation, backtesting, and reporting phases sequentially.
5. Generate a unified performance summary and ZIP archive of all reports.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Git Integration - Pull latest code from GitHub
import os
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

print("Pulling latest code from GitHub...")
os.chdir(PROJECT_ROOT)
# Initialize git and pull
!git init -b main
!git remote remove origin 2>/dev/null || true
!git remote add origin {GITHUB_REPO_URL}
!git fetch origin
!git reset --hard origin/main
print("Codebase updated to latest GitHub commit!")

In [ ]:
# Auto-install dependencies if any key package is missing
try:
    import ccxt
    import vectorbt
    import optuna
    import catboost
    import ta
    import shap
except ImportError:
    print("Some dependencies are missing. Installing from requirements.txt...")
    requirements_path = os.path.join(PROJECT_ROOT, "requirements.txt")
    !pip install -q -r {requirements_path}
    print("Installation complete!")

In [ ]:
import os
import sys

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Git Integration (Pull Latest Code from GitHub)

Set `PULL_FROM_GITHUB = True` to automatically pull the latest changes you pushed from your local development environment before running the pipeline.

In [ ]:
PULL_FROM_GITHUB = True  # Set to False if you don't want to pull changes
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

if PULL_FROM_GITHUB:
    print("Pulling latest code from GitHub...")
    os.chdir(PROJECT_ROOT)
    # Initialize git with 'main' branch to suppress warnings, and pull
    !git init -b main
    !git remote remove origin 2>/dev/null || true
    !git remote add origin {GITHUB_REPO_URL}
    # Fetch and reset/pull to match remote main branch
    !git fetch origin
    !git reset --hard origin/main
    print("Codebase updated to latest GitHub commit!")

## 3. Auto-Install Dependencies

In [ ]:
# Auto-install dependencies if any key package is missing
try:
    import ccxt
    import vectorbt
    import optuna
    import catboost
    import ta
    import shap
except ImportError:
    print("Some dependencies are missing. Installing from requirements.txt...")
    requirements_path = os.path.join(PROJECT_ROOT, "requirements.txt")
    !pip install -q -r {requirements_path}
    print("Installation complete!")

## 4. Pipeline Configuration

In [ ]:
# Define what phases to run - Set all to True for full dataset execution
RUN_DOWNLOAD = True
RUN_CLEANING = True
RUN_FEATURES = True
RUN_LABELING = True

RUN_TRAINING = True
RUN_VALIDATION = True
RUN_BACKTEST = True
RUN_EXPLAINER = True
RUN_DASHBOARD = True

## 5. Run Pipeline Phases

In [ ]:
import subprocess

def run_notebook(notebook_name):
    print(f"=== Starting {notebook_name} ===")
    path = os.path.join(PROJECT_ROOT, "notebooks", notebook_name)
    # We can use jupyter nbconvert to execute notebooks programmatically in Colab
    cmd = f"jupyter nbconvert --to notebook --execute {path} --output {path}"
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.returncode == 0:
        print(f"=== Finished {notebook_name} successfully ===\n")
    else:
        print(f"=== ERROR running {notebook_name} ===")
        print(res.stderr)
        raise RuntimeError(f"Pipeline failed at {notebook_name}")

if RUN_DOWNLOAD:
    run_notebook("01_data_download_binance.ipynb")
if RUN_CLEANING:
    run_notebook("02_data_cleaning_and_resampling.ipynb")
if RUN_FEATURES:
    run_notebook("03_feature_engineering_market_structure.ipynb")
    run_notebook("04_feature_engineering_liquidity_ict_smc.ipynb")
    run_notebook("05_feature_engineering_orderflow_volatility_sessions.ipynb")
if RUN_LABELING:
    run_notebook("06_labeling_and_event_generation.ipynb")
if RUN_TRAINING:
    run_notebook("07_model_training_baselines_and_ml.ipynb")
if RUN_VALIDATION:
    run_notebook("08_walk_forward_validation.ipynb")
if RUN_BACKTEST:
    run_notebook("09_backtesting_and_risk_management.ipynb")
if RUN_EXPLAINER:
    run_notebook("10_explainability_and_trade_journal.ipynb")
if RUN_DASHBOARD:
    run_notebook("11_dashboard_and_visual_analysis.ipynb")

print("Master Pipeline run completed successfully!")

## 6. Compile Reports & Create ZIP Archive

In [ ]:
from utils.colab_utils import generate_pipeline_summary_and_zip
from utils.config import load_global_config

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

# Aggregate all results into summary_report.txt and create reports_archive.zip
generate_pipeline_summary_and_zip(PROJECT_ROOT, symbol)

## Summary of Pipeline Run

The entire research system has been run end-to-end. All datasets, feature stores, models, backtests, and HTML dashboards are fully up-to-date and stored on Google Drive.

**Next Step**: Download the `reports_archive.zip` from your Google Drive and share it.

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()